# 02 · Análsisi exploratorio más en prof de ocupaciones

**Objetivo:** Ver si puedo relacionar las tablas por medio de los ids y qué representan las filas de cada tabla de reservas y ocuapciones.


In [ ]:
import sys; sys.path.append("..")
import pandas as pd
from src.carga import cargar_bronze

dfs = cargar_bronze()
dfs.keys()   # agencias, empresas, ocupaciones, reservaciones, segmentos_mercados

In [ ]:
oc = dfs["ocupaciones"]
res = dfs["reservaciones"]
ag = dfs["agencias"]
emp = dfs["empresas"]
seg = dfs["segmentos_mercados"]

In [ ]:
dims = ["Fecha_hoy", "ID_Agencia", "ID_Tipo_Habitacion", "ID_empresa", "ID_Segmento_Mercado"]
dup_exactos = int(oc.duplicated().sum())
dup_clave = int(oc.duplicated(subset=dims).sum())
total_r = int(len(oc))
print(f"duplicados exactos:   {dup_exactos:,}")
print(f"duplicados por clave: {dup_clave:,}")
print(f"total de registros: {total_r:,}")

In [ ]:
# Solo tiene contenido si hay duplicados por clave
oc[oc.duplicated(subset=dims, keep=False)].sort_values(dims).head(20)

Para cuando pase a silver, sumaré los ingresos por habitaciones de las que sean diferentes de 0, a continuación, quiero ver cuántas tiene 0 de ingreso por habitación:

In [ ]:
dims = ["Fecha_hoy", "ID_Agencia", "ID_Tipo_Habitacion", "ID_empresa", "ID_Segmento_Mercado", "ingresos_habitaciones"]
dup_exactos = int(oc.duplicated().sum())
dup_clave = int(oc.duplicated(subset=dims).sum())
total_r = int(len(oc))
print(f"duplicados exactos:   {dup_exactos:,}")
print(f"duplicados por clave: {dup_clave:,}")
print(f"total de registros: {total_r:,}")

In [ ]:
# Solo tiene contenido si hay duplicados por clave
oc[oc.duplicated(subset=dims, keep=False)].sort_values(dims).head(20)

# Tipos de habitación

In [ ]:
t_oc = set(oc["ID_Tipo_Habitacion"])
t_res = set(res["tipo_habitacion_id"])
print("ocupaciones:  ", len(t_oc), sorted(t_oc)[:20])
print("reservaciones:", len(t_res), sorted(t_res)[:20])
print("en común:     ", len(t_oc & t_res))

# ¿Qué es ADR? 
segpún la documentación es a daily rate, ¿eso lo puedo traducir como 

Ingresos / cuartos noche  ?

In [ ]:
# 1. Quedarse con las filas que tienen cuartos (evita dividir entre cero)
con_cuartos = oc[oc["cuartos_noche"] > 0]

# 2. Calcular el ADR con la fórmula
adr_calculado = con_cuartos["ingresos_habitaciones"] / con_cuartos["cuartos_noche"]

# 3. Diferencia con el ADR guardado en el archivo
diferencia = adr_calculado - con_cuartos["ADR"]

# 4. Dos máscaras explícitas, sin negaciones
cuadra = diferencia.abs() < 0.01
no_cuadra = diferencia.abs() >= 0.01

In [ ]:
# 5. Contar
total = len(diferencia)
cuantas_cuadran = cuadra.sum()
cuantas_no = no_cuadra.sum()

print(f"Filas comparadas: {total:,}")
print(f"Cuadran:          {cuantas_cuadran:,} ({cuantas_cuadran / total * 100:.2f} %)")
print(f"No cuadran:       {cuantas_no:,}")

Con esos resultados puedo concluir que ADR si se calcula como pensé, por lo que, para la capa silver lo recalcularé para no trabajar con ese error de las que no cuadran.

# filas sin actividad

In [ ]:
vacias = (oc["ingresos_habitaciones"] == 0) & (oc["cuartos_noche"] == 0)
print(f"Filas sin actividad: {vacias.sum():,} ({vacias.mean():.1%})")

# ¿habrán ingresos negativos? 

In [ ]:
neg = oc[oc["ingresos_habitaciones"] < 0]
pct_neg = neg["ingresos_habitaciones"].sum() / oc["ingresos_habitaciones"].sum() * 100
print(f"Filas: {len(neg):,} | suma: {neg['ingresos_habitaciones'].sum():,.2f} | % de los ingresos: {pct_neg:.2f} %")
neg["cuartos_noche"].describe()

podría eliminar esas filas o dejarlas, la verdad que no tienen un impacto grande a la hora de graficar

# Anomalías o valores extremos

In [ ]:
oc.nlargest(10, "ingresos_habitaciones")[
    ["Fecha_hoy", "ID_empresa", "ID_Segmento_Mercado", "cuartos_noche", "ingresos_habitaciones", "ADR"]]

In [ ]:
oc.loc[oc["ingresos_habitaciones"] > 0, "ingresos_habitaciones"].value_counts().head(10)

## ¿Esos extremos caben en el hotel?

**Pregunta:** ¿200 cuartos en una sola fila son físicamente posibles?

In [ ]:
valor = oc["ingresos_habitaciones"].max()
sosp = oc[oc["ingresos_habitaciones"] == valor]
sosp_cap = sosp.merge(emp[["ID_empresa", "Habitaciones_tot"]], on="ID_empresa")
sosp_cap["supera_capacidad"] = sosp_cap["cuartos_noche"] > sosp_cap["Habitaciones_tot"]
sosp_cap[["Fecha_hoy", "ID_empresa", "cuartos_noche", "Habitaciones_tot", "supera_capacidad"]]

Excluiré esas 10 filas por no ser posibles

In [ ]:
ok = oc.drop(index=sosp.index) # Excluyo las 10

# "Complementary o Cortesías"

**Pregunta:** ¿qué son las habitaciones ocupadas sin ingresos?

**mi hipótesis:** las filas con cuartos > 0 e ingresos = 0 son cortesías y está relacionado a del segmento COMPLEMENTARIES.

**Prueba:** sumar los cuartos de cortesía por segmento.

In [ ]:
cortesia = (ok["ingresos_habitaciones"] == 0) & (ok["cuartos_noche"] > 0)
cuartos_cortesia = ok.loc[cortesia, "cuartos_noche"].sum()
print(f"Cuartos noche de cortesía: {cuartos_cortesia:,} ({cuartos_cortesia / ok['cuartos_noche'].sum():.1%} del total)")
(ok[cortesia].groupby("ID_Segmento_Mercado")["cuartos_noche"].sum()
   .rename("cuartos_cortesia").reset_index()
   .merge(seg, on="ID_Segmento_Mercado"))

La verdad que acá estoy asumiendo mucho, tendría que validar con negocio, redactar en el correo a Melissa/Jorge

# ¿Qué fechas tengo para ocupaciones? 

In [ ]:
dias = ok["Fecha_hoy"].drop_duplicates().sort_values()
print("Rango:", dias.min().date(), "→", dias.max().date(), "| días distintos:", len(dias))
dias.diff().dt.days.value_counts().head()

In [ ]:
ok.groupby(ok["Fecha_hoy"].dt.to_period("M"))["cuartos_noche"].sum().plot(
    figsize=(12, 4), title="Cuartos noche por mes (ocupaciones, sin filas anómalas)")

# conclusiones de análisis exploratorio para dataset de ocupaciones: 

| Columna o patrón | Conclusión |
|---|---|
| `ADR` | correcto en el 97,2 %; lo recalcularé |
| Negativos | ajustes del ≈0,1 %; se conservan y se marcan |
| 10 filas idénticas | imposibles; se excluyen |
| Cortesías y COMPLEMENTARIES | repartidas en todos los segmentos; preguntar por correo a Melissa/Jorge |
| `Fecha_hoy` | parece ser solo fecha de registro del sistema y la tengo desde finales del 2020 hasta finales del 2024 |